In [0]:
-- ============================================================
-- DATABRICKS LAKEFLOW PIPELINE
--
-- Flow:
-- JSON files
--      ↓
-- Bronze raw data
--      ├──→ DQ issues
--      └──→ Silver valid data
--                    ↓
--               Gold summary
-- ============================================================


-- ============================================================
-- 1. BRONZE LAYER
--
-- Reads raw JSON files incrementally from the Databricks Volume.
-- Bronze keeps the source data as close to the original as possible.
-- ============================================================

CREATE OR REFRESH STREAMING TABLE bronze_transactions
COMMENT 'Raw transaction records loaded from JSON files'
AS
SELECT
    transaction_id,
    customer_id,
    transaction_date,
    amount,
    currency,
    status,

    -- Timestamp showing when the row was processed
    current_timestamp() AS ingestion_timestamp,

    -- Original source file path
    _metadata.file_path AS source_file

FROM STREAM read_files(
    '/Volumes/workspace/default/kafka_demo_files/transactions/',
    format => 'json',
    inferColumnTypes => 'true'
);


-- ============================================================
-- 2. DATA QUALITY LAYER
--
-- Stores invalid records separately for investigation.
-- Invalid records are not deleted permanently.
-- ============================================================

CREATE OR REFRESH STREAMING TABLE dq_transaction_issues
COMMENT 'Invalid transaction records with identified data quality issues'
AS
SELECT
    transaction_id,
    customer_id,
    transaction_date,
    amount,
    currency,
    status,
    ingestion_timestamp,
    source_file,

    -- Identifies the main data quality issue
    CASE
        WHEN transaction_id IS NULL
             OR TRIM(transaction_id) = ''
            THEN 'MISSING_TRANSACTION_ID'

        WHEN customer_id IS NULL
             OR TRIM(customer_id) = ''
            THEN 'MISSING_CUSTOMER_ID'

        WHEN amount IS NULL
            THEN 'MISSING_AMOUNT'

        WHEN amount <= 0
            THEN 'INVALID_AMOUNT'

        WHEN transaction_date IS NULL
             OR TRY_CAST(transaction_date AS DATE) IS NULL
            THEN 'INVALID_TRANSACTION_DATE'

        WHEN currency IS NULL
             OR TRIM(currency) = ''
            THEN 'MISSING_CURRENCY'

        WHEN status IS NULL
             OR TRIM(status) = ''
            THEN 'MISSING_STATUS'

        ELSE 'UNKNOWN_ISSUE'
    END AS dq_issue,

    current_timestamp() AS dq_checked_timestamp

FROM STREAM(bronze_transactions)

WHERE transaction_id IS NULL
   OR TRIM(transaction_id) = ''
   OR customer_id IS NULL
   OR TRIM(customer_id) = ''
   OR amount IS NULL
   OR amount <= 0
   OR transaction_date IS NULL
   OR TRY_CAST(transaction_date AS DATE) IS NULL
   OR currency IS NULL
   OR TRIM(currency) = ''
   OR status IS NULL
   OR TRIM(status) = '';


-- ============================================================
-- 3. SILVER LAYER
--
-- Keeps only valid records.
-- Standardises data types and text formatting.
-- ============================================================

CREATE OR REFRESH STREAMING TABLE silver_transactions
COMMENT 'Validated and standardised transaction records'
AS
SELECT
    TRIM(transaction_id) AS transaction_id,
    TRIM(customer_id) AS customer_id,
    CAST(transaction_date AS DATE) AS transaction_date,
    CAST(amount AS DECIMAL(18, 2)) AS amount,
    UPPER(TRIM(currency)) AS currency,
    LOWER(TRIM(status)) AS status,
    ingestion_timestamp,
    source_file

FROM STREAM(bronze_transactions)

WHERE transaction_id IS NOT NULL
  AND TRIM(transaction_id) <> ''

  AND customer_id IS NOT NULL
  AND TRIM(customer_id) <> ''

  AND amount IS NOT NULL
  AND amount > 0

  AND transaction_date IS NOT NULL
  AND TRY_CAST(transaction_date AS DATE) IS NOT NULL

  AND currency IS NOT NULL
  AND TRIM(currency) <> ''

  AND status IS NOT NULL
  AND TRIM(status) <> '';


-- ============================================================
-- 4. GOLD LAYER
--
-- Creates reporting-ready customer-level transaction metrics.
-- Only completed transactions are included.
-- ============================================================

CREATE OR REFRESH MATERIALIZED VIEW gold_customer_summary
COMMENT 'Reporting-ready transaction metrics aggregated by customer'
AS
SELECT
    customer_id,

    COUNT(*) AS transaction_count,

    SUM(amount) AS total_transaction_amount,

    ROUND(AVG(amount), 2) AS average_transaction_amount,

    MIN(transaction_date) AS first_transaction_date,

    MAX(transaction_date) AS latest_transaction_date

FROM silver_transactions

WHERE status = 'completed'

GROUP BY customer_id;